# Chapter 3 - Lab 8: <font color='blue'>Mistral AI Agent</font>

**<font color='purple'>Goal</font>**:
In this lab, you will build a **financial analysis agent using Mistral AI** that compares the P/E (Price/Earnings) ratios of two companies — Apple (AAPL) and JPMorgan (JPM) — and produces a short investment memo.

Mistral offers an OpenAI-compatible function-calling interface but with its own SDK and model lineup. In this lab you will build the agent loop **by hand** — sending messages, checking for tool calls, executing the requested tool, and appending the result — which is illuminating because it shows you exactly what frameworks like LangChain and AutoGen automate for you.

This is the same reference task used across every framework lab in Chapter 3 — comparing all of them on the *same* task makes the differences in API style, abstractions, and ergonomics easy to spot.

**<font color='purple'>Tech stack</font>**:

* **Mistral SDK** (`mistralai`) — `Mistral` client with chat + tools.
* **`mistral-large-latest`** — Mistral's flagship reasoning model.
* **Manual agent loop** — explicit iteration until the model stops asking for tools.

You will need an OpenAI API key with some credits available.

## 1. Install packages

Install the framework and its dependencies.

In [ ]:
%pip install -q mistralai pydantic python-dotenv

## 2. Set up the API key(s)

If you are running this notebook in **Google Colab**, add your OpenAI key in the left vertical menu (the *key* icon) under the name `OPENAI_API_KEY`.

If you are running locally, replace the cell below with `os.environ['OPENAI_API_KEY'] = '...'` or load it from a `.env` file.

In [ ]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['MISTRAL_API_KEY'] = userdata.get('MISTRAL_API_KEY')

## 3. Bootstrap the shared task setup

Every framework lab in this chapter shares the same task, tools, finance dataset, and prompts. These are factored out into `common.py`. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%203/common.py',
        'common.py',
    )

from common import (
    get_stock_data,
    compute_pe,
    finance_data,
    system_message,
    input_message,
)

print('Tools loaded. Reference task:')
print(' ', input_message)

## 4. Define the tool schemas (OpenAI-compatible JSON)

Mistral uses the same JSON schema format as OpenAI for tools. We declare two — `get_stock_data` and `compute_pe_ratio` — with their parameter signatures.

In [ ]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_stock_data',
            'description': 'Get stock data including price and EPS for a given ticker symbol',
            'parameters': {
                'type': 'object',
                'properties': {
                    'ticker': {'type': 'string', 'description': 'Ticker symbol (e.g., AAPL, JPM)'},
                },
                'required': ['ticker'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'compute_pe_ratio',
            'description': 'Compute P/E ratio given stock price and earnings per share',
            'parameters': {
                'type': 'object',
                'properties': {
                    'price': {'type': 'number', 'description': 'Stock price'},
                    'eps':   {'type': 'number', 'description': 'Earnings per share'},
                },
                'required': ['price', 'eps'],
            },
        },
    },
]

## 5. Tool executor

A small dispatcher maps a tool name to the matching Python function. This is exactly what framework runtimes do for you under the hood.

In [ ]:
def execute_tool(name: str, args: dict) -> str:
    if name == 'get_stock_data':
        r = get_stock_data(args['ticker'])
        return f"Stock data for {args['ticker']}: Price=${r.price}, EPS=${r.eps}"
    if name == 'compute_pe_ratio':
        return f"P/E ratio: {compute_pe(args['price'], args['eps'])}"
    return f'Unknown tool: {name}'

## 6. The manual agent loop

We send the conversation to Mistral, append any tool calls and their results, and loop until the model stops asking for tools. Cap the iterations to avoid run-aways.

In [ ]:
import os, json
from mistralai import Mistral

client = Mistral(api_key=os.environ['MISTRAL_API_KEY'])

messages = [
    {'role': 'system', 'content': system_message},
    {'role': 'user',   'content': input_message},
]

for iteration in range(5):
    response = client.chat.complete(
        model='mistral-large-latest',
        messages=messages,
        tools=tools,
        tool_choice='auto',
        temperature=0,
    )
    msg = response.choices[0].message
    messages.append({
        'role': 'assistant',
        'content': msg.content,
        'tool_calls': msg.tool_calls,
    })
    if not msg.tool_calls:
        break
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments)
        result = execute_tool(tc.function.name, args)
        messages.append({
            'role': 'tool',
            'name': tc.function.name,
            'content': result,
            'tool_call_id': tc.id,
        })

print(msg.content)

## 7. Results

You should see the agent call `get_stock_data` twice (once per ticker), then call `compute_pe` twice to compute each P/E ratio, and finally produce a short memo comparing the two.

**What to notice about Mistral AI (hand-rolled agent loop) specifically:**

* Writing the loop by hand demystifies what every other framework does — *send messages, dispatch tool calls, append results, repeat*.
* This is the most code in the chapter for a reason: it shows the floor that abstractions stand on.
* If you find yourself customising agent-runtime behaviour heavily, dropping to a hand-rolled loop like this is sometimes the right call — you get total control.
* Trade-off: every change (a new tool, a new termination rule, observability) is more code in your repository instead of a config change.